# Inference

In this new notebook, we'll cover the process of generatinga nd ranking predictions on theoretical "new" data. Due to time and resource constraints, the data was synthetically generated. The inference process should be:

1. Load pipeline
2. Load data
3. Run pipeline on data to get probabilties
4. Append probabilities to data
5. Sort by probabilities and select top k

### Setup

In [33]:
# Import libraries
import joblib
import pandas as pd

In [38]:
# Load model and data
xgb_pipeline = joblib.load("../artifacts/best_xgb_pipeline.joblib")
dataset_sizes = ["small", "medium", "large"]

# Change index to load different size inference datasets
inference_df = pd.read_csv(f"../data/external/inference/demo_{dataset_sizes[0]}_inference.csv")

# Create flight_id column
inference_df = inference_df.reset_index(names = "flight_id")

### Inference

In [40]:
# Calculate probabilites and append to df
delay_probs = xgb_pipeline.predict_proba(inference_df)[:,1]

inference_df["prob_delayed"] = delay_probs

In [42]:
inference_df[["flight_id", "prob_delayed"]].head()

,flight_id,prob_delayed
0,0,0.253817
1,1,0.335163
2,2,0.150066
3,3,0.540712
4,4,0.487818


In [43]:
inference_df.columns

Index(['flight_id', 'Origin', 'CRSElapsedTime', 'AirTime', 'Distance', 'Year',
       'IsWeekend', 'Season', 'CRSArr_TimeOfDay', 'CRSDep_TimeOfDay',
       'dest_temperature_2m_mean', 'dest_wind_speed_10m_max',
       'dest_wind_gusts_10m_max', 'dest_precipitation_sum', 'dest_rain_sum',
       'dest_snowfall_sum', 'dest_precipitation_hours', 'dest_weather_code',
       'dest_temperature_2m_range', 'origin_temperature_2m_mean',
       'origin_wind_speed_10m_max', 'origin_wind_gusts_10m_max',
       'origin_precipitation_sum', 'origin_rain_sum', 'origin_snowfall_sum',
       'origin_precipitation_hours', 'origin_weather_code',
       'origin_temperature_2m_range', 'airline_name', 'SinMonth', 'CosMonth',
       'SinDay', 'CosDay', 'prob_delayed'],
      dtype='object')

### Display

In [45]:
import tabulate

headers = ["Flight ID", "Origin Airport", "Airline", "Probability of Delayed Arrival"]
rows = inference_df[["flight_id", "Origin", "airline_name", "prob_delayed"]].values.tolist()

top_k = 10
rows_sorted = sorted(rows, key = lambda x: x[3], reverse = True)[:top_k]

print(tabulate.tabulate(rows_sorted, headers, tablefmt = "simple"))

  Flight ID  Origin Airport    Airline                   Probability of Delayed Arrival
-----------  ----------------  ----------------------  --------------------------------
         13  DFW               American Airlines Inc.                          0.704584
          8  BNA               Southwest Airlines Co.                          0.584756
          3  HOU               Southwest Airlines Co.                          0.540712
         14  MKE               PSA Airlines Inc.                               0.54029
         10  MCO               Frontier Airlines Inc.                          0.534635
          4  BGR               PSA Airlines Inc.                               0.487818
          7  MKE               PSA Airlines Inc.                               0.397107
         12  RSW               American Airlines Inc.                          0.344618
          1  CVG               PSA Airlines Inc.                               0.335163
          9  DCA               PS